# 🚀 QuantDataPipeline — 量化數據中台

**全自動一鍵執行**：輸入參數 → 按下播放鍵 → 自動下載 + Greeks 計算 + 多週期特徵聚合

---

### 📋 使用說明
1. 在下方表單填入 **FinMind API Token**
2. 設定 **每小時 API 額度** (帳號等級對應的 requests/hour)
3. 選擇 **GitHub 分支**、**回溯天數** (填 `0` = 全量抓取到 2011-01-03)
4. 按下左側 ▶️ 播放鍵即可

> ⚠️ 首次執行會自動安裝依賴套件（約 30 秒），後續執行會直接跳過。
>
> ⚠️ 2019-01-16 ~ 2019-06-30 期間部分資料不完整 (FinMind 官方已知缺失)。

In [ ]:
#@title 🎛️ QuantDataPipeline 控制面板 { run: "auto", display-mode: "form" }
#@markdown ---
#@markdown ### 🔑 API 設定
FINMIND_API_TOKEN = '' #@param {type:"string"}
API_QUOTA_PER_HOUR = 1600 #@param {type:"integer"}
#@markdown ---
#@markdown ### 🌿 GitHub 分支
BRANCH = '3' #@param {type:"string"}
#@markdown ---
#@markdown ### ⚙️ 管線參數
LOOKBACK_DAYS = 30 #@param {type:"integer"}
#@markdown > `0` = 全量抓取 (從 2011-01-03 至今)。資料區間: 2011-01-03 ~ now
SKIP_GREEKS = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### 💾 儲存設定
SYNC_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_PATH = '/content/drive/MyDrive/QuantData' #@param {type:"string"}
#@markdown ---

# ═══════════════════════════════════════════════════════════════
# 以下為自動執行邏輯，不需修改
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, os, time, shutil, json, logging, math, importlib
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, HTML

# ── 清除舊模組快取 (Colab 重複執行時必須) ──
stale_prefixes = ['core.', 'fetchers.', 'processors.', 'storage.', 'compute_greeks']
for mod_name in list(sys.modules.keys()):
    if any(mod_name.startswith(p) or mod_name == p.rstrip('.') for p in stale_prefixes):
        del sys.modules[mod_name]

# ── 設定環境變數 (在 import config 之前) ──
os.environ['FINMIND_API_TOKEN'] = FINMIND_API_TOKEN
os.environ['FINMIND_QUOTA_PER_HOUR'] = str(API_QUOTA_PER_HOUR)

# ── 隱藏底層 logger 冗長輸出 ──
for name in ['pipeline', 'pipeline.retry', 'pipeline.extractor',
             'pipeline.orchestrator', 'pipeline.http', 'pipeline.rate_limiter',
             'pipeline.schema']:
    logging.getLogger(name).setLevel(logging.CRITICAL)

# ── 鎖定輸出高度 + 自動捲軸 ──
display(HTML("""
<style>
  .output_scroll { height: 420px !important; overflow-y: auto !important; }
  .output_wrapper { max-height: 420px !important; overflow-y: auto !important; }
  .output_area pre { font-family: 'Fira Code', 'Consolas', monospace; font-size: 13px; line-height: 1.6; }
</style>
<script>
  (function() {
    var output = document.querySelector('.output_scroll, .output_wrapper');
    if (output) { output.style.maxHeight = '420px'; output.style.overflowY = 'auto'; }
    var observer = new MutationObserver(function() {
      var el = document.querySelector('.output_scroll, .output_wrapper');
      if (el) el.scrollTop = el.scrollHeight;
    });
    var target = document.querySelector('.output_area');
    if (target) observer.observe(target, {childList: true, subtree: true});
  })();
</script>
"""))

# ── 列印工具 ──
api_call_count = 0
pipeline_start_time = time.time()

def log(icon, msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"{icon} {ts} | {msg}", flush=True)

def header(title):
    print(f"\n{'━'*55}", flush=True)
    print(f"  {title}", flush=True)
    print(f"{'━'*55}", flush=True)

# ── 永久性錯誤偵測 ──
FATAL_KEYWORDS = ['user level', 'update your user level', 'please upgrade',
                  'permission denied', 'invalid token', 'unauthorized', 'sponsor']
def is_fatal(e):
    return any(kw in str(e).lower() for kw in FATAL_KEYWORDS)

# ── 日期計算 ──
DATA_EARLIEST = '2011-01-03'
today = datetime.now()
end_date = today.strftime('%Y-%m-%d')

if LOOKBACK_DAYS <= 0:
    start_date = DATA_EARLIEST
    lookback_label = f'全量 ({DATA_EARLIEST} ~ {end_date})'
else:
    start_date = (today - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    lookback_label = f'{LOOKBACK_DAYS} 天 ({start_date} ~ {end_date})'

# ── API 用量預估 ──
CALLS_PER_DAY = 2
rate_delay = round(3600 / max(API_QUOTA_PER_HOUR, 1) * 1.1, 2)
total_calendar_days = (today - datetime.strptime(start_date, '%Y-%m-%d')).days
est_trading_days = int(total_calendar_days * 0.66)
est_api_calls = est_trading_days * CALLS_PER_DAY + 1
est_minutes = round(est_api_calls * rate_delay / 60, 1)
est_hours = round(est_minutes / 60, 1)

header('🚀 QuantDataPipeline 啟動中')
log('📅', f'範圍: {lookback_label}')
log('🔧', f'Greeks: {"開" if not SKIP_GREEKS else "關"} | Drive: {"開" if SYNC_TO_DRIVE else "關"} | 分支: {BRANCH}')
print(flush=True)
log('📊', f'═══ API 用量預估 ═══')
log('  ', f'額度:      {API_QUOTA_PER_HOUR} 次/hr')
log('  ', f'速率:      {rate_delay}s / 請求')
log('  ', f'預估交易日: ~{est_trading_days} 天')
log('  ', f'預估呼叫數: ~{est_api_calls} 次')
if est_hours >= 1:
    log('  ', f'預估耗時:  ~{est_hours} 小時')
else:
    log('  ', f'預估耗時:  ~{est_minutes} 分鐘')

# ═══════════════════════════════════════════════════════════════
# Phase 0: 環境準備
# ═══════════════════════════════════════════════════════════════
header('📦 Phase 0: 環境準備')

REPO_URL = 'https://github.com/hsp1234-web/SP_OP_20260220.git'
REPO_DIR = Path('/content/SP_OP_20260220')

if SYNC_TO_DRIVE:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        log('✅', 'Google Drive 已掛載')
    except Exception as e:
        log('⚠️', f'Drive 掛載失敗: {e}')
        SYNC_TO_DRIVE = False

if REPO_DIR.exists():
    log('🔄', f'更新代碼 (分支 {BRANCH})...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=str(REPO_DIR), capture_output=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), capture_output=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=str(REPO_DIR), capture_output=True)
else:
    log('📥', f'下載代碼 (分支 {BRANCH})...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], capture_output=True)

PROJECT_DIR = REPO_DIR / 'QuantDataPipeline'
# 確保路徑在最前面
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

log('📦', '安裝依賴套件...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'polars', 'numba', 'scipy', 'numpy', 'requests', 'python-dotenv'],
    capture_output=True, text=True
)
log('✅', '依賴套件就緒')

# 寫入 .env
env_path = PROJECT_DIR / '.env'
env_path.write_text(f'FINMIND_API_TOKEN={FINMIND_API_TOKEN}\nFINMIND_QUOTA_PER_HOUR={API_QUOTA_PER_HOUR}\n')
log('🔑', f'API Token: {"已設定" if FINMIND_API_TOKEN else "未設定 (匿名模式)"}')

# ═══════════════════════════════════════════════════════════════
# Phase 1: 資料下載
# ═══════════════════════════════════════════════════════════════
header('⚡ Phase 1: 資料下載')

os.chdir(str(PROJECT_DIR))

# 現在 import 一定是最新的代碼 (前面已清除快取)
from core.config import DATA_DIR, DB_PATH, RATE_LIMIT_DELAY
from core.db_metadata_manager import DBManager
from core.fetch_orchestrator import process_task, seed_tasks_from_dates
from fetchers.datasets.technical import trading_date
from fetchers.infrastructure.http_session import get_session

log('⏱️', f'實際速率: {RATE_LIMIT_DELAY}s / 請求')

drive_data = Path(DRIVE_PATH) if SYNC_TO_DRIVE else None

if SYNC_TO_DRIVE and drive_data:
    drive_db = drive_data / 'status.db'
    if drive_db.exists():
        shutil.copy2(drive_db, DB_PATH)
        log('📋', '已從 Drive 還原 status.db')

DBManager._reset_instance()
db = DBManager(DB_PATH)
session = get_session()

log('📅', f'日期範圍: {start_date} → {end_date}')

actual_calls = 0
try:
    trading_dates_df = trading_date.fetch_trading_dates(session, start_date, end_date)
    api_call_count += 1
    if trading_dates_df is None or trading_dates_df.is_empty():
        log('⚠️', '此範圍內無交易日')
        trading_dates_df = None
    else:
        n_dates = len(trading_dates_df)
        actual_calls = n_dates * CALLS_PER_DAY + 1
        actual_time = round(actual_calls * RATE_LIMIT_DELAY / 60, 1)
        log('✅', f'找到 {n_dates} 個交易日')
        if actual_time >= 60:
            log('📊', f'需要 {actual_calls} 次 API 呼叫 (~{round(actual_time/60,1)} 小時)')
        else:
            log('📊', f'需要 {actual_calls} 次 API 呼叫 (~{actual_time} 分鐘)')
except Exception as e:
    if is_fatal(e):
        log('⛔', f'API 權限不足: {str(e)[:60]}')
        log('💡', f'請設定 FINMIND_API_TOKEN (需 backer/sponsor 等級)')
        trading_dates_df = None
    else:
        log('❌', f'取得交易日失敗: {e}')
        trading_dates_df = None

TARGET_DATASETS = [('TaiwanOptionTick', 'TXO'), ('TaiwanFuturesTick', 'TX')]
COOLDOWN = 300
pipeline_aborted = False

if trading_dates_df is not None:
    seed_tasks_from_dates(db, trading_dates_df, TARGET_DATASETS)
    pending = db.get_pending_tasks()
    total = len(pending)
    log('📊', f'待下載任務: {total} 個 (已完成的自動跳過)')

    done = 0
    for i, (task_id, tdate, dset, did) in enumerate(pending, 1):
        try:
            process_task(task_id, tdate, dset, did)
            api_call_count += 1
            status = db.get_task_status(task_id)
            if status == 3:
                log('➖', f'{tdate} {did:>3} | 無資料跳過  [{api_call_count}/{actual_calls}]')
            elif status == 1:
                log('✅', f'{tdate} {did:>3} | 下載成功  [{api_call_count}/{actual_calls}]')
                done += 1
            else:
                log('⏳', f'{tdate} {did:>3} | 待重試  [{api_call_count}/{actual_calls}]')
        except Exception as e:
            api_call_count += 1
            if is_fatal(e):
                log('⛔', f'帳號權限不足，管線中止')
                log('💡', f'請設定 FINMIND_API_TOKEN (需 backer/sponsor 等級)')
                pipeline_aborted = True
                break
            err = str(e).lower()
            if any(k in err for k in ['429', 'rate limit', 'quota', 'too many']):
                log('🧊', f'API 限速 — 冷卻 {COOLDOWN}s...')
                time.sleep(COOLDOWN)
                try:
                    process_task(task_id, tdate, dset, did)
                    api_call_count += 1
                    log('✅', f'{tdate} {did:>3} | 重試成功  [{api_call_count}/{actual_calls}]')
                    done += 1
                except:
                    log('❌', f'{tdate} {did:>3} | 重試失敗')
            else:
                log('❌', f'{tdate} {did:>3} | {str(e)[:40]}')

    if not pipeline_aborted:
        log('📊', f'Phase 1 完成: {done}/{total} 個任務成功')

# ═══════════════════════════════════════════════════════════════
# Phase 2: Greeks 計算
# ═══════════════════════════════════════════════════════════════
if not SKIP_GREEKS and not pipeline_aborted:
    header('🧮 Phase 2: Greeks 特徵計算')

    from compute_greeks_pipeline import compute_greeks_for_date

    l1_tasks = db.get_tasks_by_status(1)
    opt_dates = set()
    for tid, tdate, dset, did in l1_tasks:
        if dset == 'TaiwanOptionTick':
            year = tdate.split('-')[0]
            gpath = DATA_DIR / year / 'GreeksFeatures' / f'TXO_Greeks_{tdate}.parquet'
            if not gpath.exists():
                fut_tid = f'{tdate}_TaiwanFuturesTick_TX'
                fut_st = db.get_task_status(fut_tid)
                if fut_st and fut_st >= 1:
                    opt_dates.add(tdate)

    computable = sorted(opt_dates)
    total_c = len(computable)
    log('📊', f'待計算日期: {total_c} 個 (純本地計算，不耗 API)')

    done_c = 0
    for i, d in enumerate(computable, 1):
        try:
            df, opath, ok = compute_greeks_for_date(d)
            if ok:
                db.update_task_status(f'{d}_TaiwanOptionTick_TXO', 2)
                log('✅', f'{d} | Greeks 完成 ({len(df):,} 筆) [{i}/{total_c}]')
                done_c += 1
            else:
                log('➖', f'{d} | 無有效資料 [{i}/{total_c}]')
        except Exception as e:
            log('❌', f'{d} | {str(e)[:50]}')

    log('📊', f'Phase 2 完成: {done_c}/{total_c} 個日期成功')

# ═══════════════════════════════════════════════════════════════
# Phase 3: Drive 同步
# ═══════════════════════════════════════════════════════════════
if SYNC_TO_DRIVE and drive_data and not pipeline_aborted:
    header('☁️ Phase 3: Google Drive 同步')

    drive_data.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DB_PATH, drive_data / 'status.db')
    log('✅', 'status.db 已同步')

    synced = 0
    if DATA_DIR.exists():
        for pq in DATA_DIR.rglob('*.parquet'):
            rel = pq.relative_to(DATA_DIR)
            dest = drive_data / 'data' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or dest.stat().st_size != pq.stat().st_size:
                shutil.copy2(pq, dest)
                synced += 1
    log('✅', f'已同步 {synced} 個 Parquet 到 Drive')

# ═══════════════════════════════════════════════════════════════
# 完成統計
# ═══════════════════════════════════════════════════════════════
elapsed_sec = time.time() - pipeline_start_time
elapsed_min = round(elapsed_sec / 60, 1)

header('🏁 執行完畢' if not pipeline_aborted else '⛔ 管線已中止')

if pipeline_aborted:
    log('⛔', '因帳號權限不足而中止：')
    log('  ', '1. TaiwanOptionTick / TaiwanFuturesTick 需 backer 或 sponsor 等級')
    log('  ', '2. 請在表單頂部填入有效的 FINMIND_API_TOKEN')
    log('  ', '3. 填入 Token 後重新執行此儲存格即可')
else:
    stats = {
        '待處理': len(db.get_tasks_by_status(0)),
        'L1完成': len(db.get_tasks_by_status(1)),
        'L2完成': len(db.get_tasks_by_status(2)),
        '已跳過': len(db.get_tasks_by_status(3)),
    }
    total_all = sum(stats.values())
    actual_rate = round(api_call_count / max(elapsed_sec / 3600, 0.001), 0)

    log('📊', f'═══ 執行統計 ═══')
    log('  ', f'API 呼叫: {api_call_count} 次')
    log('  ', f'實際速率: {actual_rate:.0f} 次/hr (額度: {API_QUOTA_PER_HOUR}/hr)')
    log('  ', f'總耗時:   {elapsed_min} 分鐘')
    print(flush=True)
    log('📊', f'═══ 任務統計 ═══')
    for k, v in stats.items():
        bar = '█' * int(v / max(total_all, 1) * 20)
        log('  ', f'{k}: {v:4d} {bar}')
    log('🎉', '管線執行完畢！下次執行會自動跳過已完成的任務。')
